# Integrated YOLOv8 Training Pipeline for Electronic Schematics

This notebook consolidates the data pipeline and model training steps for the Electronic Schematic Symbol Classifier. It's designed for continuity across Colab sessions and ease of deployment, ensuring the YOLOv8 model always has access to its data.

In [2]:
# Standard Library Imports
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path
import json
from collections import defaultdict
import glob

# Third-party Imports
from google.colab import drive
from PIL import Image
from sklearn.model_selection import train_test_split

# Mount Google Drive to access persisted data
drive.mount('/content/drive')

print("All necessary libraries imported and Google Drive mounted.")

Mounted at /content/drive
All necessary libraries imported and Google Drive mounted.


## 1. Data Setup: Load or Generate YOLOv8 Dataset

This section handles getting the data ready for YOLOv8 training. It prioritizes loading a pre-processed YOLO dataset (`digitize_hcd_yolo_dataset.zip`) from Google Drive for efficiency. If this processed dataset isn't found in Drive, it will automatically run the full data pipeline to generate it from the raw data (`digitize_hcd.zip`).

In [5]:
# --- Import paths ---
from pathlib import Path


# --- Define Paths and Names ---

# Path to the original raw dataset zip in Google Drive
digitize_hcd_raw_zip_path_in_drive = "/content/drive/MyDrive/AI4ALL/data/digitize_hcd.zip"

# Name for the pre-processed YOLO dataset zip (to be saved/loaded from Drive)
yolo_dataset_zip_name = "digitize_hcd_yolo_dataset.zip"

# Path in Google Drive where the processed YOLO dataset zip will be stored/loaded from
drive_output_path = "/content/drive/MyDrive/AI4ALL/data/"

# Local working directories
output_path = Path("/content/data") # Where raw data will be unzipped
yolo_data_dir = Path("/content/data_yolo") # Where YOLO-formatted data will reside

# Ensure output directories exist
output_path.mkdir(parents=True, exist_ok=True)
yolo_data_dir.mkdir(parents=True, exist_ok=True)

# Path for the processed YOLO dataset zip in Google Drive
yolo_dataset_zip_path_in_drive = Path(drive_output_path) / yolo_dataset_zip_name
yolo_dataset_local_zip_path = Path("/tmp") / yolo_dataset_zip_name


# --- Conditional Data Loading Logic ---

if yolo_dataset_zip_path_in_drive.exists():
    print(f"Found pre-processed YOLO dataset zip in Drive: {yolo_dataset_zip_path_in_drive}")
    print("Unzipping pre-processed dataset to /content/data_yolo...")
    shutil.copy(yolo_dataset_zip_path_in_drive, yolo_dataset_local_zip_path)
    with zipfile.ZipFile(yolo_dataset_local_zip_path, "r") as archive:
        archive.extractall(yolo_data_dir)
    print("Pre-processed YOLO dataset loaded successfully.")

else:
    print("Pre-processed YOLO dataset not found in Drive. Running full data pipeline...")

    # 1. Unzip raw data
    print(f"Unzipping raw dataset from {digitize_hcd_raw_zip_path_in_drive} to {output_path}...")
    with zipfile.ZipFile(digitize_hcd_raw_zip_path_in_drive, "r") as archive:
        archive.extractall(output_path)
    print(f"Contents of {output_path}: {os.listdir(output_path)}")

    # 2. Create YOLO dataset directory structure
    images_dir = yolo_data_dir / "images"
    labels_dir = yolo_data_dir / "labels"

    for split in ['train', 'val']:
      (images_dir / split).mkdir(parents=True, exist_ok=True)
      (labels_dir / split).mkdir(parents=True, exist_ok=True)
    print("Created YOLO dataset directory structure.")

    # 3. Define Helper Functions (Image Resizing and COCO to YOLO Conversion)
    def resize_images(source_dir, dest_dir, max_size=1280):
        images = glob.glob(os.path.join(source_dir, "*.jpg")) + glob.glob(os.path.join(source_dir, "*.png"))
        for img_path in images:
            img = Image.open(img_path)
            img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
            filename = os.path.basename(img_path)
            img.save(os.path.join(dest_dir, filename))

    def convert_coco_to_yolo(coco_json_path, labels_out_dir):
        with open(coco_json_path, 'r') as f:
            coco_data = json.load(f)

        categories = {cat['id']: cat['name'] for cat in coco_data.get('categories', [])}
        cat_to_yolo = {cat_id: i for i, cat_id in enumerate(categories.keys())}

        os.makedirs(labels_out_dir, exist_ok=True)

        with open(os.path.join(labels_out_dir, '_classes.txt'), 'w') as f:
            for cat_id, name in categories.items():
                f.write(f"{cat_to_yolo[cat_id]}: {name}\n")

        images = {img['id']: img for img in coco_data.get('images', [])}
        ann_by_img = defaultdict(list)
        for ann in coco_data.get('annotations', []):
            ann_by_img[ann['image_id']].append(ann)

        for img_id, img_info in images.items():
            img_w, img_h = img_info['width'], img_info['height']
            base_name = os.path.splitext(os.path.basename(img_info['file_name']))[0]
            txt_path = os.path.join(labels_out_dir, f"{base_name}.txt")
            anns = ann_by_img.get(img_id, [])

            with open(txt_path, 'w') as out_f:
                for ann in anns:
                    x_min, y_min, w, h = ann['bbox']
                    class_id = cat_to_yolo[ann['category_id']]
                    x_center = (x_min + w / 2.0) / img_w
                    y_center = (y_min + h / 2.0) / img_h
                    w_norm = w / img_w
                    h_norm = h / img_h

                    x_center = max(0.0, min(1.0, x_center))
                    y_center = max(0.0, min(1.0, y_center))
                    w_norm = max(0.0, min(1.0, w_norm))
                    h_norm = max(0.0, min(1.0, h_norm))

                    out_f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")
        print(f"Successfully converted annotations for {len(images)} images.")


    # 4. Process COCO data to YOLO format
    source_data_root = str(output_path)
    source_annotation_dir = os.path.join(source_data_root, "Component Symbol and Text Label Data")
    coco_json_path = os.path.join(source_annotation_dir, "component_annotations.json")
    source_images_base = os.path.join(source_annotation_dir, "Circuit Diagram Images")
    temp_yolo_labels_out_dir = "/tmp/yolo_labels_temp"
    os.makedirs(temp_yolo_labels_out_dir, exist_ok=True)

    convert_coco_to_yolo(coco_json_path, temp_yolo_labels_out_dir)

    with open(coco_json_path, 'r') as f:
        coco_data = json.load(f)
    image_filenames = [img['file_name'] for img in coco_data.get('images', [])]
    train_filenames, val_filenames = train_test_split(image_filenames, test_size=0.2, random_state=42)

    # 5. Copy files to YOLO directory structure
    def copy_files(filenames, img_dest_dir, label_dest_dir, source_img_base, temp_label_src_dir):
        for filename in filenames:
            src_image_path = os.path.join(source_img_base, filename)
            dest_image_path = os.path.join(img_dest_dir, filename)
            shutil.copy(src_image_path, dest_image_path)

            base_name = os.path.splitext(filename)[0]
            src_label_path = os.path.join(temp_label_src_dir, base_name + ".txt")
            dest_label_path = os.path.join(label_dest_dir, base_name + ".txt")

            if os.path.exists(src_label_path):
                shutil.copy(src_label_path, dest_label_path)
            else:
                with open(dest_label_path, 'w') as f:
                    pass # Create empty label file for negative samples

    copy_files(train_filenames, images_dir / "train", labels_dir / "train", source_images_base, temp_yolo_labels_out_dir)
    copy_files(val_filenames, images_dir / "val", labels_dir / "val", source_images_base, temp_yolo_labels_out_dir)

    shutil.rmtree(temp_yolo_labels_out_dir)
    print("Data successfully moved and structured for YOLO training!")

    # 6. Zip and save the processed YOLO dataset to Google Drive
    os.makedirs(drive_output_path, exist_ok=True) # Ensure destination in Drive exists
    shutil.make_archive(str(yolo_dataset_local_zip_path.parent / yolo_dataset_local_zip_path.stem), 'zip', yolo_data_dir)
    shutil.copy(yolo_dataset_local_zip_path, yolo_dataset_zip_path_in_drive)
    print(f"YOLO dataset successfully zipped and saved to Drive: {yolo_dataset_zip_path_in_drive}")

Found pre-processed YOLO dataset zip in Drive: /content/drive/MyDrive/AI4ALL/data/digitize_hcd_yolo_dataset.zip
Unzipping pre-processed dataset to /content/data_yolo...
Pre-processed YOLO dataset loaded successfully.


In [6]:
# Create or update data.yaml for YOLOv8

dataset_dir = str(yolo_data_dir)

yaml_content = f"""
train: {yolo_data_dir}/images/train
val: {yolo_data_dir}/images/val

nc: 17
names: ['BJT-NPN', 'BJT-PNP', 'Capacitor', 'Diode', 'GND', 'I-AC', 'I-DC', 'Inductor', 'MOSFET-N', 'MOSFET-P', 'Op-Amp', 'Resistor', 'V-AC', 'V-DC', 'V-DC (one port)', 'Wire Crossover', 'Zener Diode']
"""

# Write data.yaml to the YOLO data directory
with open(os.path.join(dataset_dir, "data.yaml"), "w") as f:
    f.write(yaml_content.strip())

print("data.yaml created successfully in the YOLO data directory.")

data.yaml created successfully in the YOLO data directory.


## 2. Model Training Configuration

This section configures and initiates the YOLOv8 model training. It uses a robust set of parameters suitable for the given dataset and hardware constraints.

In [7]:
# Install Ultralytics (if not already installed)
%pip install ultralytics

# Now import YOLO after ensuring it's installed
from ultralytics import YOLO

# Ensure data.yaml exists (this check should now pass due to previous steps)
data_yaml_path = os.path.join(str(yolo_data_dir), "data.yaml")
print(f"Checking for {data_yaml_path}...")
if not os.path.exists(data_yaml_path):
    print("Error: data.yaml still not found. Something went wrong in data preparation.")
else:
    print(f"data.yaml found at {data_yaml_path}.")

# Initialize the YOLOv8 Nano model
model = YOLO('yolov8n.pt')

# Advanced Training Loop
results = model.train(
    data=data_yaml_path,
    epochs=100,           # Train for up to 100 epochs
    patience=10,          # Early stopping if validation mAP doesn't improve for 10 epochs
    batch=16,             # Safe batch size for T4 GPU (use 8 if OOM occurs)
    imgsz=640,            # Standard image size
    device=0,             # Force GPU usage
    project='AI4ALL_Runs',# Directory to save results
    name='schematic_v1',  # Experiment name

    # --- Data Augmentation for Hand-Drawn Schematics ---
    degrees=15.0,         # Slight rotation (+/- 15 degrees)
    scale=0.5,            # Scale variations (zoom in/out)
    hsv_s=0.2,            # Saturation adjustments (handles different ink/paper)
    hsv_v=0.2,            # Value/Brightness adjustments
    fliplr=0.5,           # Horizontal flip (OK for circuits)
    flipud=0.0            # NO vertical flip (keeps symbols upright)
)

print("Training complete! Best weights saved to AI4ALL_Runs/schematic_v1/weights/best.pt")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Checking for /content/data_yolo/data.yaml...
data.yaml found at /content/data_yolo/data.yaml.
Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_yolo/data.yaml, degrees=15.0

## 3. Model Evaluation

After training, the model's performance is evaluated on the validation set to assess its accuracy (mAP, Precision, Recall).

In [ ]:
# Evaluate the model on the validation set
metrics = model.val()

print(f"Mean Average Precision (mAP50-95): {metrics.box.map}")
print(f"Mean Average Precision (mAP50): {metrics.box.map50}")

## 4. Inference & Export for Deployment

Finally, the trained model is exported to a format suitable for deployment (e.g., ONNX), which is often preferred for web applications like Streamlit due to faster inference times.

In [ ]:
# Export the model to ONNX format for faster inference in the web backend
export_path = model.export(format='onnx', imgsz=640)
print(f"Model exported successfully to: {export_path}")

# Optional: Run inference on a test image (uncomment and replace 'path/to/test.jpg' with a real image)
# results = model('/content/data_yolo/images/val/your_test_image.jpg', save=True)

In [ ]:
import shutil
import os

# Define the source path of the exported ONNX model
source_onnx_path = export_path # 'export_path' variable is available from the previous cell

# Define the destination path in Google Drive
destination_onnx_dir = os.path.join(drive_output_path, 'exported_models') # Using 'drive_output_path' from previous cells

# Ensure the destination directory exists in Drive
os.makedirs(destination_onnx_dir, exist_ok=True)

# Define the full destination path for the ONNX model in Drive
destination_onnx_path = os.path.join(destination_onnx_dir, os.path.basename(source_onnx_path))

# Copy the ONNX model to Google Drive
shutil.copy(source_onnx_path, destination_onnx_path)

print(f"ONNX model copied from '{source_onnx_path}' to '{destination_onnx_path}' for persistence.")